# Notebook 1 — YOLO11m-seg (one-stage, CNN)

**Model:** YOLO11m-seg dari Ultralytics, pretrained COCO (`yolo11m-seg.pt`).
**Keluarga:** one-stage, CNN. Mask dibentuk dari kombinasi linear *prototype mask* beresolusi 1/4 input
(160×160 untuk input 640) dengan koefisien per objek, lalu dipotong dengan box.

Augmentasi dan hyperparameter memakai default Ultralytics (mosaic, HSV, flip, scale), sama untuk ketiga fraksi.
Model terbaik dipilih dari *fitness* validasi Ultralytics.

**Cara menjalankan di Kaggle:**
1. *Settings* → *Accelerator*: **GPU T4 x2**, *Internet*: **On**.
2. *Add Input* → *Notebooks* → pilih output **Notebook 0** (berisi `indoor_seg_data.zip`).
3. **Pilot dulu** (`PILOT = True`, jalankan interaktif): 2 epoch pada data 100% untuk mengukur waktu per epoch.
   Notebook mencetak saran `EPOCHS` agar run 100% selesai dalam `TARGET_HOURS_100` jam.
4. Isi `EPOCHS`, set `PILOT = False`, lalu **Save Version → Save & Run All** (berjalan di background, maks 12 jam).

**Cara kerja dengan 2 GPU:** setiap run memakai satu GPU dan berjalan sebagai proses terpisah. GPU0 menjalankan run 100%,
GPU1 menjalankan run 25% lalu 50%. Batch size, learning rate, dan jumlah epoch identik untuk ketiga fraksi, jadi
satu-satunya yang berbeda antar run adalah jumlah data latih.

**Evaluasi:** semua model dievaluasi dengan fungsi pycocotools yang sama (`seg_common.py`, identik di Notebook 1–3)
pada split `valid`, `test`, dan `test_own` (kalau sudah ada), maksimal 100 prediksi per gambar.

**Output** (`/kaggle/working`): `runs/<model>_<fraksi>/` (bobot, `results.csv`, prediksi, evaluasi),
`metrics_<model>.csv`, `per_class_<model>.csv`, `gpu_time_log.csv`, dan `logs/`.

**Test set buatan sendiri yang datang belakangan:** jalankan ulang Notebook 0 dengan `OWN_TEST_*` terisi, lalu
jalankan notebook ini dengan `RUN_TRAINING = False` dan output versi training notebook ini ditambahkan sebagai input.
Bobot lama dipakai, tidak ada training ulang.

In [1]:
!pip install -q ultralytics pycocotools

## 1. Konfigurasi

In [2]:
import glob
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
import yaml

MODEL_KEY = "yolo11m_seg"
MODEL_LABEL = "YOLO11m-seg"
FAMILY = "one-stage, CNN"
PRETRAINED = "yolo11m-seg.pt"

EPOCHS = 150         # pilot: 2 epoch = 1,4 mnt (~0,7 mnt/epoch) -> saran 173; dipakai 150, cadangan untuk run paralel
BATCH = 16           # per GPU (satu run = satu GPU)
IMGSZ = 640
WORKERS = 2

TAGS = ["025", "050", "100"]          # fraksi data latih (NRP digit 3)
SEED = 42
MAX_HOURS_PER_RUN = 3.0               # pengaman: run berhenti rapi setelah epoch yang melewati batas ini

PILOT = False                         # True: 2 epoch pada data 100% untuk mengukur waktu per epoch (sudah dijalankan)
PILOT_EPOCHS = 2
TARGET_HOURS_100 = 2.0                # target durasi run 100% untuk saran jumlah epoch
RUN_TRAINING = True                   # False: lewati training, pakai bobot dari output versi sebelumnya

DATA_ZIP_GLOB = "/kaggle/input/**/indoor_seg_data.zip"
PREV_RUNS_GLOB = "/kaggle/input/**/runs"
DATA_ROOT = Path("/tmp/data")
WORK = Path("/kaggle/working")
RUNS = WORK / "runs"
POLL_SECONDS = 120                    # interval laporan progres

## 2. Data dari Notebook 0

In [3]:
zips = sorted(glob.glob(DATA_ZIP_GLOB, recursive=True))
assert zips, "indoor_seg_data.zip tidak ditemukan. Tambahkan output Notebook 0 lewat Add Input."
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(DATA_ROOT)

# path absolut di YAML YOLO disamakan dengan lokasi ekstrak
for y in DATA_ROOT.glob("data_*.yaml"):
    cfg = yaml.safe_load(y.read_text(encoding="utf-8"))
    cfg["path"] = DATA_ROOT.as_posix()
    y.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")

INFO = json.loads((DATA_ROOT / "dataset_info.json").read_text(encoding="utf-8"))
EVAL_SPLITS = [s for s in ["valid", "test", "test_own"] if (DATA_ROOT / "annotations" / f"{s}.json").exists()]
print("zip    :", zips[0])
print("kelas  :", [c["name"] for c in INFO["classes"]])
print("subset :", INFO["subsets"])
print("evaluasi pada:", EVAL_SPLITS)
if "test_own" not in EVAL_SPLITS:
    print("-> test_own belum ada. Setelah Notebook 0 dijalankan ulang dengan test set buatanmu, "
          "jalankan notebook ini dengan RUN_TRAINING = False.")

zip    : /kaggle/input/notebooks/sultandaulay/cv-is-data-prep/indoor_seg_data.zip
kelas  : ['backpack', 'bed', 'blanket', 'book', 'bottle', 'bowl', 'box', 'cabinet-merged', 'cardboard', 'ceiling-merged', 'cell phone', 'chair', 'clock', 'couch', 'counter', 'cup', 'curtain', 'dining table', 'door-stuff', 'fire-extinguisher', 'floor-other-merged', 'floor-wood', 'flower', 'food-other-merged', 'fork', 'hair drier', 'handbag', 'keyboard', 'knife', 'laptop', 'light', 'microwave', 'mirror-stuff', 'mouse', 'oven', 'pan', 'paper', 'paper-merged', 'person', 'pillow', 'plate', 'plug', 'potted plant', 'refrigerator', 'remote', 'rug-merged', 'scissor', 'scissors', 'shelf', 'sink', 'spoon', 'st-', 'stairs', 'stove-merged', 'table-merged', 'teddy bear', 'toaster', 'toilet', 'toothbrush', 'towel', 'toy', 'trash-bin', 'tree-merged', 'tv', 'vase', 'wall', 'wall-brick', 'wall-other-merged', 'wall-stone', 'wall-tile', 'wall-wood', 'wallet', 'window-blind', 'window-other']
subset : {'025': {'images': 99, 'i

## 3. Kode training dan evaluasi

Ditulis ke file supaya setiap run berjalan sebagai proses terpisah di GPU-nya sendiri. `seg_common.py` identik di Notebook 1–3.

In [4]:
%%writefile seg_common.py
"""Fungsi bersama ketiga notebook training: seed, encode mask, evaluasi COCO (pycocotools), log CSV."""
import contextlib
import csv
import io
import json
import random
from pathlib import Path

import numpy as np
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

STATS = ["AP", "AP50", "AP75", "APs", "APm", "APl"]


def set_seed(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def write_json(obj, path, indent=None):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent)


def append_csv(path, row, fields):
    path = Path(path)
    new = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", restval="")
        if new:
            w.writeheader()
        w.writerow(row)


def encode_mask(mask):
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return rle


def det(image_id, category_id, mask, score, bbox=None):
    """Satu prediksi dalam format hasil COCO. bbox (x, y, w, h) dihitung dari mask kalau tidak diberikan."""
    rle = encode_mask(mask)
    if bbox is None:
        bbox = mask_utils.toBbox(rle).tolist()
    return {"image_id": int(image_id), "category_id": int(category_id), "segmentation": rle,
            "bbox": [float(v) for v in bbox], "score": float(score)}


def coco_evaluate(gt_json, dets):
    """Mask AP dan box AP gaya COCO, plus mask AP per kelas.
    Untuk 'segm', bbox dibuang dari prediksi supaya area dihitung dari mask (sama seperti Detectron2)."""
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(gt_json))
    metrics, per_class = {}, {}
    for iou_type in ["segm", "bbox"]:
        if not dets:
            metrics.update({f"{iou_type}_{s}": 0.0 for s in STATS})
            continue
        d = [{k: v for k, v in x.items() if not (iou_type == "segm" and k == "bbox")} for x in dets]
        with contextlib.redirect_stdout(io.StringIO()):
            ev = COCOeval(gt, gt.loadRes(d), iou_type)
            ev.evaluate()
            ev.accumulate()
            ev.summarize()
        metrics.update({f"{iou_type}_{s}": float(v) for s, v in zip(STATS, ev.stats[:6])})
        if iou_type == "segm":
            prec = ev.eval["precision"][:, :, :, 0, -1]   # IoU x recall x kelas, area=all, maxDets=100
            for k, cat_id in enumerate(ev.params.catIds):
                p = prec[:, :, k]
                p = p[p > -1]
                per_class[gt.cats[cat_id]["name"]] = float(p.mean()) if p.size else float("nan")
    return metrics, per_class


def evaluate_and_save(gt_json, dets, out_dir, split, extra=None):
    metrics, per_class = coco_evaluate(gt_json, dets)
    write_json(dets, Path(out_dir) / f"preds_{split}.json")
    result = {"split": split, **(extra or {}), **metrics, "per_class_segm_AP": per_class}
    write_json(result, Path(out_dir) / f"eval_{split}.json", indent=2)
    print(f"[{split}] mask AP={metrics['segm_AP']:.4f}  AP50={metrics['segm_AP50']:.4f}  "
          f"box AP={metrics['bbox_AP']:.4f}  ({len(dets)} prediksi)", flush=True)
    return result

Writing seg_common.py


In [5]:
%%writefile train_yolo.py
"""Training & prediksi YOLO-seg (Ultralytics) untuk satu fraksi data. Satu proses, satu GPU."""
import argparse
import time
from pathlib import Path

import cv2
import numpy as np

from seg_common import det, evaluate_and_save, read_json, write_json


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("mode", choices=["train", "predict"])
    p.add_argument("--tag", required=True)
    p.add_argument("--name", required=True)
    p.add_argument("--data-root", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--weights", default="yolo11m-seg.pt")
    p.add_argument("--epochs", type=int, default=80)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--max-hours", type=float, default=0)
    p.add_argument("--splits", nargs="+", default=["valid", "test"])
    a = p.parse_args()
    a.out = str(Path(a.out).resolve())     # path relatif akan dipindah Ultralytics ke runs/segment/...
    return a


def train(a):
    from ultralytics import YOLO

    model = YOLO(a.weights)
    t0, state = time.time(), {"stopped_by_time": False}

    def time_guard(trainer):
        if a.max_hours and (time.time() - t0) / 3600 > a.max_hours and not trainer.stop:
            trainer.stop = state["stopped_by_time"] = True
            print(f"[time_guard] melewati {a.max_hours} jam, berhenti setelah epoch {trainer.epoch + 1}", flush=True)

    model.add_callback("on_fit_epoch_end", time_guard)
    model.train(data=str(Path(a.data_root) / f"data_{a.tag}.yaml"), epochs=a.epochs, imgsz=a.imgsz,
                batch=a.batch, device=0, workers=a.workers, seed=a.seed, project=a.out, name=a.name,
                exist_ok=True, patience=a.epochs + 1, plots=True)
    run = Path(model.trainer.save_dir)
    expected = Path(a.out) / a.name
    assert run.resolve() == expected.resolve(), f"Ultralytics menyimpan run di {run}, bukan {expected}"
    epochs_done = sum(1 for _ in open(run / "results.csv")) - 1
    write_json({"tag": a.tag, "epochs_planned": a.epochs, "epochs_done": epochs_done,
                "stopped_by_time": state["stopped_by_time"], "train_hours": round((time.time() - t0) / 3600, 4),
                "train_images": len(Path(a.data_root, f"train_{a.tag}.txt").read_text().split())},
               run / "train_summary.json", indent=2)


def predict(a):
    from ultralytics import YOLO

    run, root = Path(a.out) / a.name, Path(a.data_root)
    model = YOLO(str(run / "weights" / "best.pt"))
    for split in a.splits:
        gt_json = root / "annotations" / f"{split}.json"
        if not gt_json.exists():
            print(f"[{split}] tidak ada, dilewati")
            continue
        images = read_json(gt_json)["images"]
        dets, t_inf = [], 0.0
        for i, im in enumerate(images):
            t0 = time.time()
            r = model.predict(str(root / "images" / split / im["file_name"]), imgsz=a.imgsz, conf=0.001, iou=0.7,
                              max_det=100, retina_masks=True, device=0, verbose=False)[0]
            if r.masks is None or len(r.boxes) == 0:
                continue
            h, w = im["height"], im["width"]
            masks = r.masks.data.cpu().numpy() > 0.5
            if masks.shape[1:] != (h, w):
                masks = np.stack([cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                                  for m in masks]) > 0
            xyxy = r.boxes.xyxy.cpu().numpy()
            cls = r.boxes.cls.cpu().numpy().astype(int)
            conf = r.boxes.conf.cpu().numpy()
            if i > 0:                                   # gambar pertama = pemanasan GPU, tidak dihitung
                t_inf += time.time() - t0
            for m, b, c, s in zip(masks, xyxy, cls, conf):
                if m.any():
                    dets.append(det(im["id"], c + 1, m, s, [b[0], b[1], b[2] - b[0], b[3] - b[1]]))
        ms = 1000 * t_inf / max(len(images) - 1, 1)
        evaluate_and_save(gt_json, dets, run, split, {"model": "yolo", "tag": a.tag, "ms_per_image": round(ms, 2)})


if __name__ == "__main__":
    args = parse()
    train(args) if args.mode == "train" else predict(args)

Writing train_yolo.py


## 4. Bobot pretrained dan runner

In [6]:
from ultralytics import YOLO
from ultralytics.utils.checks import check_amp, check_font

# Unduh bobot (dan file cek AMP) sekali di sini, supaya dua proses paralel tidak mengunduh bersamaan.
_m = YOLO(PRETRAINED)
WEIGHTS_PATH = Path(_m.ckpt_path).resolve()
check_font("Arial.ttf")
if torch.cuda.is_available():
    check_amp(_m.model.cuda())
del _m
torch.cuda.empty_cache()
print("bobot:", WEIGHTS_PATH)

SCRIPT = Path("train_yolo.py").resolve()


def train_cmd(tag, epochs, name):
    return [sys.executable, SCRIPT, "train", "--tag", tag, "--name", name, "--data-root", DATA_ROOT, "--out", RUNS,
            "--weights", WEIGHTS_PATH, "--epochs", epochs, "--batch", BATCH, "--imgsz", IMGSZ,
            "--workers", WORKERS, "--seed", SEED, "--max-hours", MAX_HOURS_PER_RUN]


def predict_cmd(tag, name):
    return [sys.executable, SCRIPT, "predict", "--tag", tag, "--name", name, "--data-root", DATA_ROOT,
            "--out", RUNS, "--imgsz", IMGSZ, "--splits", *EVAL_SPLITS]

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
bobot: /kaggle/working/yolo11m-seg.pt


In [7]:
import re

LOG_DIR = WORK / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
GPU_LOG = WORK / "gpu_time_log.csv"
N_GPU = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPU)]
print(f"{N_GPU} GPU:", GPU_NAMES)
_lock = threading.Lock()


def job(kind, name, cmd):
    return {"kind": kind, "name": name, "cmd": [str(c) for c in cmd], "log": str(LOG_DIR / f"{name}_{kind}.log")}


def _last_line(path):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 4000))
            text = f.read().decode("utf-8", "ignore")
    except FileNotFoundError:
        return ""
    parts = [p.strip() for p in re.split(r"[\r\n]+", text) if p.strip()]
    return parts[-1][:170] if parts else ""


def _run_job(j, gpu):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    start, t0 = datetime.now(), time.time()
    with open(j["log"], "w", encoding="utf-8") as log:
        log.write("$ " + " ".join(j["cmd"]) + "\n")
        log.flush()
        rc = subprocess.run(j["cmd"], env=env, stdout=log, stderr=subprocess.STDOUT, cwd=WORK).returncode
    row = {"model": MODEL_KEY, "job": j["name"], "kind": j["kind"], "gpu_index": gpu,
           "gpu_name": GPU_NAMES[gpu] if gpu < N_GPU else "cpu",
           "start": start.isoformat(timespec="seconds"), "end": datetime.now().isoformat(timespec="seconds"),
           "hours": round((time.time() - t0) / 3600, 4), "returncode": rc}
    with _lock:
        pd.DataFrame([row]).to_csv(GPU_LOG, mode="a", header=not GPU_LOG.exists(), index=False)
        status = "OK" if rc == 0 else f"GAGAL (kode {rc}), lihat {j['log']}"
        print(f"[{row['end']}] {j['kind']} {j['name']} di GPU{gpu}: {row['hours']:.2f} jam, {status}", flush=True)
    return rc


def run_jobs(jobs):
    """Dengan 2 GPU: job pertama di GPU0, sisanya berurutan di GPU1 (dua jalur paralel).
    Dengan 1 GPU: semua berurutan di GPU0. Setiap job = satu proses terpisah dengan satu GPU."""
    lanes = {0: jobs[:1], 1: jobs[1:]} if N_GPU >= 2 and len(jobs) > 1 else {0: jobs}
    results, started = {}, set()

    def lane(gpu, lane_jobs):
        for j in lane_jobs:
            started.add(j["name"])
            results[j["name"]] = _run_job(j, gpu)

    threads = [threading.Thread(target=lane, args=(g, js)) for g, js in lanes.items() if js]
    for t in threads:
        t.start()
    t0 = last = time.time()
    while any(t.is_alive() for t in threads):
        time.sleep(2)
        if time.time() - last >= POLL_SECONDS:
            last = time.time()
            for j in jobs:
                if j["name"] in started and j["name"] not in results:
                    print(f"  {(time.time() - t0) / 60:6.1f} mnt | {j['name']}: {_last_line(j['log'])}", flush=True)
    for t in threads:
        t.join()
    return results

2 GPU: ['Tesla T4', 'Tesla T4']


## 5. Training (atau pilot)

In [8]:
order = sorted(TAGS, key=lambda t: (t != "100", t))          # run 100% di jalur GPU0, 025 lalu 050 di GPU1
run_name = lambda tag: f"{MODEL_KEY}_{tag}"

if PILOT:
    name = f"{MODEL_KEY}_100_pilot"
    run_jobs([job("pilot", name, train_cmd("100", PILOT_EPOCHS, name))])
    hours = pd.read_csv(GPU_LOG).query("job == @name")["hours"].iloc[-1]
    per_epoch_min = hours * 60 / PILOT_EPOCHS
    print(f"\nPilot: {PILOT_EPOCHS} epoch dalam {hours * 60:.1f} menit -> ~{per_epoch_min:.1f} menit/epoch "
          f"(termasuk waktu muat model, jadi sedikit lebih besar dari aslinya)")
    print(f"Saran EPOCHS agar run 100% selesai dalam ~{TARGET_HOURS_100} jam: "
          f"{max(1, int(TARGET_HOURS_100 * 60 / per_epoch_min))}")
    print("Langkah berikutnya: isi EPOCHS, set PILOT = False, lalu Save Version -> Save & Run All.")
elif RUN_TRAINING:
    rc = run_jobs([job("train", run_name(t), train_cmd(t, EPOCHS, run_name(t))) for t in order])
    rows = []
    for t in TAGS:
        p = RUNS / run_name(t) / "train_summary.json"
        rows.append(json.loads(p.read_text()) if p.exists() else {"tag": t, "status": "GAGAL, cek log"})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    found = [p for p in glob.glob(PREV_RUNS_GLOB, recursive=True)
             if all((Path(p) / run_name(t)).exists() for t in TAGS)]
    assert found, f"Folder runs berisi {[run_name(t) for t in TAGS]} tidak ditemukan di {PREV_RUNS_GLOB}"
    shutil.copytree(found[0], RUNS, dirs_exist_ok=True)
    print("Memakai bobot dari", found[0])

     2.0 mnt | yolo11m_seg_100:       5/150      9.79G      0.681      1.552      1.622      1.102          0        398        640: 36% ━━━━──────── 9/25 1.4it/s 6.8s<11.5s
     2.0 mnt | yolo11m_seg_025:      10/150      9.96G     0.6972      1.499      1.735      1.089          0        397        640: 42% ━━━━━─────── 3/7 1.1s/it 2.8s<4.3s
     4.0 mnt | yolo11m_seg_100:      10/150        10G     0.6157      1.297      1.202       1.04          0        243        640: 100% ━━━━━━━━━━━━ 25/25 1.5it/s 16.4s
     4.0 mnt | yolo11m_seg_025:                  Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 16% ━━────────── 1/6 2.3s/it 0.7s
     6.0 mnt | yolo11m_seg_100:      16/150      9.82G     0.5654      1.172      1.002      1.005          0        446        640: 40% ━━━━╸─────── 10/25 1.5it/s 7.3s<10.3s
     6.0 mnt | yolo11m_seg_025:                  Class     Images  Instances      Box(P          R      mAP5

## 6. Evaluasi pada valid, test, dan test_own

In [9]:
if not PILOT:
    run_jobs([job("predict", run_name(t), predict_cmd(t, run_name(t))) for t in order
              if (RUNS / run_name(t)).exists()])

    rows, per_class = [], {}
    for t in TAGS:
        summ_p = RUNS / run_name(t) / "train_summary.json"
        summ = json.loads(summ_p.read_text()) if summ_p.exists() else {}
        for s in EVAL_SPLITS:
            p = RUNS / run_name(t) / f"eval_{s}.json"
            if not p.exists():
                continue
            r = json.loads(p.read_text())
            per_class[(t, s)] = r.pop("per_class_segm_AP")
            rows.append({"model": MODEL_LABEL, "family": FAMILY, "fraksi": int(t), "split": s,
                         **{k: round(v, 4) for k, v in r.items() if k.startswith(("segm_", "bbox_"))},
                         "ms_per_image": r.get("ms_per_image"), "train_hours": summ.get("train_hours"),
                         "epochs_done": summ.get("epochs_done"), "stopped_by_time": summ.get("stopped_by_time")})
    metrics = pd.DataFrame(rows)
    metrics.to_csv(WORK / f"metrics_{MODEL_KEY}.csv", index=False)
    pc = pd.DataFrame(per_class).round(4)
    pc.columns = [f"{t}_{s}" for t, s in pc.columns]
    pc.to_csv(WORK / f"per_class_{MODEL_KEY}.csv")

    print("Mask AP (COCO AP@[.50:.95]) per fraksi data latih:")
    print(metrics.pivot(index="fraksi", columns="split", values="segm_AP").reindex(columns=EVAL_SPLITS).to_string(), "\n")
    print("Mask AP50:")
    print(metrics.pivot(index="fraksi", columns="split", values="segm_AP50").reindex(columns=EVAL_SPLITS).to_string(), "\n")
    print("Mask AP per kelas:")
    print(pc.to_string())

     2.0 mnt | yolo11m_seg_100: [valid] mask AP=0.2706  AP50=0.3596  box AP=0.3004  (10087 prediksi)
     2.0 mnt | yolo11m_seg_025: [valid] mask AP=0.1696  AP50=0.2443  box AP=0.1922  (11442 prediksi)
[2026-11-15T08:55:18] predict yolo11m_seg_100 di GPU0: 0.04 jam, OK
[2026-11-15T08:55:25] predict yolo11m_seg_025 di GPU1: 0.04 jam, OK
     4.0 mnt | yolo11m_seg_050: [valid] mask AP=0.1963  AP50=0.2743  box AP=0.2182  (11482 prediksi)
[2026-11-15T08:57:43] predict yolo11m_seg_050 di GPU1: 0.04 jam, OK
Mask AP (COCO AP@[.50:.95]) per fraksi data latih:
split    valid    test
fraksi                
25      0.1696  0.0968
50      0.1963  0.1017
100     0.2706  0.1029 

Mask AP50:
split    valid    test
fraksi                
25      0.2443  0.1470
50      0.2743  0.1532
100     0.3596  0.1561 

Mask AP per kelas:
                    025_valid  025_test  050_valid  050_test  100_valid  100_test
backpack               0.0000    0.0276     0.0000    0.1386     0.0000    0.0768
bed           

## 7. Ringkasan waktu GPU dan output

In [10]:
if GPU_LOG.exists():
    log = pd.read_csv(GPU_LOG)
    print(log.to_string(index=False))
    print("\nTotal jam-GPU per jenis job:")
    print(log.groupby("kind")["hours"].sum().round(2).to_string())

print("\nOutput penting di", WORK)
for p in sorted(WORK.glob("*.csv")):
    print("  ", p.name)
for p in sorted(RUNS.glob("*/")):
    files = [f.name for f in p.iterdir()]
    print("  ", p.relative_to(WORK), "->", ", ".join(sorted(files))[:150])

      model             job    kind  gpu_index gpu_name               start                 end  hours  returncode
yolo11m_seg yolo11m_seg_025   train          1 Tesla T4 2026-11-15T07:51:52 2026-11-15T08:17:07 0.4209           0
yolo11m_seg yolo11m_seg_100   train          0 Tesla T4 2026-11-15T07:51:52 2026-11-15T08:46:28 0.9102           0
yolo11m_seg yolo11m_seg_050   train          1 Tesla T4 2026-11-15T08:17:07 2026-11-15T08:53:00 0.5980           0
yolo11m_seg yolo11m_seg_100 predict          0 Tesla T4 2026-11-15T08:53:00 2026-11-15T08:55:18 0.0383           0
yolo11m_seg yolo11m_seg_025 predict          1 Tesla T4 2026-11-15T08:53:00 2026-11-15T08:55:25 0.0401           0
yolo11m_seg yolo11m_seg_050 predict          1 Tesla T4 2026-11-15T08:55:25 2026-11-15T08:57:43 0.0384           0

Total jam-GPU per jenis job:
kind
predict    0.12
train      1.93

Output penting di /kaggle/working
   gpu_time_log.csv
   metrics_yolo11m_seg.csv
   per_class_yolo11m_seg.csv
   runs/yolo11m_s

## Catatan untuk logbook dan laporan

Isi setelah notebook selesai (bagian 4 tugas: bukti proses):

- **Waktu GPU:** salin tabel dari `gpu_time_log.csv` (jam per job, GPU yang dipakai).
- **Log training asli:** `runs/<run>/results.csv` untuk setiap fraksi.
- **Keputusan:** jumlah epoch dan alasannya (hasil pilot), batch size, ukuran input, apakah ada run yang berhenti karena batas waktu (`stopped_by_time`).
- **Kendala:** error, OOM, atau hal yang diubah selama pengerjaan, beserta solusinya.
- **Bantuan AI:** bagian kode yang dibuat dengan AI dan kasus saat saran AI salah atau perlu diperbaiki (bagian 5 tugas).